# Oxta Contábil — Continued Pre-Training (NSOS C++ nativo)

**Objetivo:** Treinar o runtime NSOS nativo (C++ 1.58-bit) com os datasets brasileiros que baixamos. Sem Q&A formatting, sem ChatML, sem SFT — só **texto cru → próximo token**.

**Datasets usados (todos públicos, comercial-OK):**
1. **CulturaX PT-BR** — texto web brasileiro (sample, filtrado por keywords fiscais)
2. **BR-TaxQA-R** — perguntas+respostas IRPF + **173MB de acórdãos CARF** (extraídos como texto cru)
3. **BACEN FAQ** — vocabulário financeiro do Banco Central
4. **LeNER-Br** — texto jurídico brasileiro

**Diferença para o smoke notebook anterior:**

| | Smoke (gpt2-small-portuguese) | Esse (NSOS nativo) |
|--|--|--|
| Modelo | 125M HF Transformers | Nosso C++ runtime, 40-80M, 1.58-bit |
| Treino | SFT em Q&A (ChatML) | Continued pre-training (texto cru) |
| Saída | checkpoint .bin HF | NSOS pack (.nsos, ~10-200MB) |
| Roda em prod? | não, é só baseline | sim, vai pro Oxta API |

**Tempo estimado:** ~20-40 min em T4 (build NSOS + download + treino smoke ~500-2000 steps).

## 1. Mount Drive (caching de builds + checkpoints)

Sem Drive cada session re-builda nsos_ext do zero (~10 min perdidos). Com Drive a segunda vez é instantâneo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repo (branch `oxta-contabil`)

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/vitorGgC569/reimagined.git'
BRANCH = 'oxta-contabil'
REPO_ROOT = Path('/content/reimagined-main')

if not REPO_ROOT.exists():
    print(f'Cloning {REPO_URL} (branch={BRANCH})...')
    subprocess.run(['git', 'clone', '--depth=1', '--branch', BRANCH,
                    REPO_URL, str(REPO_ROOT)], check=True)
else:
    print(f'Repo já existe em {REPO_ROOT}, fazendo pull...')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'reset', '--hard', f'origin/{BRANCH}'], check=True)

os.environ['REPO_ROOT'] = str(REPO_ROOT)
os.chdir(REPO_ROOT)
print(f'\ncwd: {os.getcwd()}')
subprocess.run(['git', 'log', '--oneline', '-3'], check=True)

## 3. Bootstrap — build nsos_ext (cached via Drive)

Primeira vez: ~10 min de build. Sessions seguintes: ~5 segundos (cópia do .so do Drive).

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / 'colab'))

# Use o param-scale 40m por default; trocar pra '80m' aumenta proporcionalmente custo de compute
os.environ['NSOS_PARAM_SCALE'] = os.environ.get('NSOS_PARAM_SCALE', '40m')

import colab_bootstrap
env = colab_bootstrap.setup_environment(verbose=True)

BUILD_DIR = Path(env['build_dir'])
EXT_SO = Path(env['ext_so'])
PROFILE = env['profile']
GPU_NAME = env['gpu_name']

print(f'\nBuild OK. Profile detected: {PROFILE} on {GPU_NAME}')
print(f'nsos_ext.so:   {EXT_SO}')

In [ ]:
# Adiciona o diretório do .so ao sys.path e importa
sys.path.insert(0, str(BUILD_DIR))

import importlib
if 'nsos_ext' in sys.modules:
    del sys.modules['nsos_ext']
import nsos_ext

print(f'nsos_ext loaded from: {nsos_ext.__file__}')
print(f'\nAvailable top-level symbols (first 20):')
for name in sorted(dir(nsos_ext))[:20]:
    if not name.startswith('_'):
        print(f'  {name}')

## 4. Download datasets brasileiros

Roda o script `download_datasets.py` que criamos. Baixa CulturaX PT-BR (subset), BR-TaxQA-R, BACEN FAQ, LeNER-Br. Skip `br_doc_extraction` aqui porque é imagens — não serve pra pre-training de texto.

In [ ]:
# Instalar deps do script
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'datasets>=2.16', 'huggingface_hub', 'tqdm'], check=True)

# Cache em Drive pra não re-baixar a cada session
DATA_ROOT = Path('/content/drive/MyDrive/oxta_contabil_data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# CulturaX é o único grande — limita a 3GB pra caber em sessão e dar pra iterar
# Em produção: --target-bytes 50GB ou mais
CULTURAX_TARGET = '3GB'

cmd = [
    sys.executable,
    str(REPO_ROOT / 'OXN/nsos/scripts/oxta_contabil/download_datasets.py'),
    '--output-dir', str(DATA_ROOT),
    '--include', 'culturax_ptbr,br_taxqa,bacen_faq,lener_br',
    '--target-bytes', CULTURAX_TARGET,
    '--culturax-shards', '20',
]
print(f'Running: {" ".join(cmd)}\n')
result = subprocess.run(cmd, check=False)
print(f'\nexit code: {result.returncode}')

In [ ]:
# Verifica o que foi baixado
RAW_DIR = DATA_ROOT / 'oxta_contabil' / 'raw'
print(f'Datasets em {RAW_DIR}:\n')
for d in sorted(RAW_DIR.iterdir()):
    if d.is_dir():
        size_mb = sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / 1024**2
        n_files = sum(1 for f in d.rglob('*') if f.is_file())
        print(f'  {d.name:30s} {size_mb:>8.1f} MB  ({n_files} files)')

# Manifest summary
import json
manifest_path = DATA_ROOT / 'oxta_contabil' / 'manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print(f'\nManifest results:')
    for name, r in manifest['results'].items():
        status = r.get('status', '?')
        print(f'  {name:30s} status={status}')

## 5. Converter para corpus de texto cru

Cada dataset tem formato diferente. Vamos extrair só os campos de texto e gerar um único JSONL com `{"text": "..."}`. O trainer NSOS lê esse formato direto.

**Importante:** sem Q&A wrapping, sem ChatML, sem system prompts. É texto puro pra modelo aprender vocabulário e estrutura de linguagem fiscal/jurídica brasileira.

In [ ]:
import json
import glob
from typing import Iterator

CORPUS_PATH = DATA_ROOT / 'oxta_contabil' / 'corpus.jsonl'
MIN_TEXT_LEN = 200  # filtra fragmentos micro

def extract_culturax_ptbr(root: Path) -> Iterator[str]:
    """CulturaX foi salvo como .jsonl shards pelo download_datasets.
    Cada linha tem {text, is_fiscal}. Yields raw text."""
    for shard in sorted(root.glob('culturax_ptbr_shard_*.jsonl')):
        with open(shard, encoding='utf-8') as f:
            for line in f:
                try:
                    row = json.loads(line)
                except json.JSONDecodeError:
                    continue
                text = (row.get('text') or '').strip()
                if len(text) >= MIN_TEXT_LEN:
                    yield text

def extract_br_taxqa(root: Path) -> Iterator[str]:
    """BR-TaxQA-R tem 3 arquivos JSON: questions, refs, acordaos CARF.
    Todos viram texto cru (sem distinção Q/A — só junta)."""
    # 1. Questions: junta question_text + answer como parágrafos
    questions_files = sorted(root.glob('questions_QA_*.json'))
    if questions_files:
        data = json.loads(questions_files[-1].read_text(encoding='utf-8'))
        items = data['questions'] if isinstance(data, dict) and 'questions' in data \
                else data if isinstance(data, list) else list(data.values())
        for item in items:
            if not isinstance(item, dict):
                continue
            q = (item.get('question_text') or item.get('question') or '').strip()
            a = item.get('answer') or item.get('answer_cleaned') or ''
            if isinstance(a, list):
                a = '\n'.join(str(x) for x in a)
            a = str(a).strip()
            text = (q + '\n\n' + a).strip() if q and a else (q or a)
            if len(text) >= MIN_TEXT_LEN:
                yield text

    # 2. Referred legal documents: full text de leis citadas
    refs_files = sorted(root.glob('referred_legal_documents_*.json'))
    if refs_files:
        data = json.loads(refs_files[-1].read_text(encoding='utf-8'))
        items = data if isinstance(data, list) else list(data.values()) if isinstance(data, dict) else []
        for item in items:
            if not isinstance(item, dict):
                continue
            text = (item.get('filedata') or item.get('text') or '').strip()
            if len(text) >= MIN_TEXT_LEN:
                yield text

    # 3. Acórdãos CARF: maior fonte (~173MB) — texto técnico-fiscal real
    carf_files = sorted(root.glob('acordaos_CARF_*.json'))
    if carf_files:
        data = json.loads(carf_files[-1].read_text(encoding='utf-8'))
        items = data if isinstance(data, list) else list(data.values()) if isinstance(data, dict) else []
        for item in items:
            if isinstance(item, dict):
                text = (item.get('text') or item.get('filedata') or '').strip()
                if len(text) >= MIN_TEXT_LEN:
                    yield text

def extract_bacen_faq(root: Path) -> Iterator[str]:
    """BACEN FAQ: hugging face dataset salvo via load_dataset.save_to_disk."""
    try:
        from datasets import load_from_disk
        ds = load_from_disk(str(root))
    except Exception as e:
        print(f'  bacen_faq não carregou via load_from_disk: {e}')
        return
    # Campos comuns: question, answer, body, text
    for row in ds:
        parts = []
        for key in ('question', 'pergunta', 'q'):
            if key in row and row[key]:
                parts.append(str(row[key]).strip())
                break
        for key in ('answer', 'resposta', 'a', 'body', 'text'):
            if key in row and row[key]:
                parts.append(str(row[key]).strip())
                break
        text = '\n\n'.join(p for p in parts if p)
        if len(text) >= MIN_TEXT_LEN:
            yield text

def extract_lener_br(root: Path) -> Iterator[str]:
    """LeNER-Br: CoNLL-style NER (token, tag).  Junta tokens em texto cru."""
    try:
        from datasets import load_from_disk
        ds = load_from_disk(str(root))
    except Exception as e:
        print(f'  lener_br não carregou via load_from_disk: {e}')
        return
    for row in ds:
        tokens = row.get('tokens') or row.get('words') or []
        if tokens:
            text = ' '.join(str(t) for t in tokens).strip()
            if len(text) >= MIN_TEXT_LEN:
                yield text

EXTRACTORS = {
    'culturax_ptbr':     extract_culturax_ptbr,
    'br_taxqa':          extract_br_taxqa,
    'bacen_faq':         extract_bacen_faq,
    'lener_br':          extract_lener_br,
}

stats = {}
total_chars = 0
total_docs = 0

with open(CORPUS_PATH, 'w', encoding='utf-8') as out:
    for name, extractor in EXTRACTORS.items():
        dataset_dir = RAW_DIR / name
        if not dataset_dir.exists():
            print(f'[skip] {name} — diretório não existe')
            continue
        n, chars = 0, 0
        for text in extractor(dataset_dir):
            out.write(json.dumps({'text': text, 'source': name}, ensure_ascii=False) + '\n')
            n += 1
            chars += len(text)
        stats[name] = {'docs': n, 'chars': chars}
        total_docs += n
        total_chars += chars
        print(f'[ok] {name:18s} {n:>8d} docs  {chars/1024**2:>8.1f} MB texto')

print(f'\n{"="*60}')
print(f'CORPUS TOTAL: {total_docs:,} docs, {total_chars/1024**2:.1f} MB texto')
print(f'Salvo em: {CORPUS_PATH}')
print(f'Tamanho do corpus.jsonl: {CORPUS_PATH.stat().st_size/1024**2:.1f} MB')

## 6. Inicializar NSOS engine

Cria um modelo do zero com configuração mamba_small (40M params) ou carrega checkpoint existente.

In [ ]:
# Configuração do modelo
config = nsos_ext.ModelConfig()
print('ModelConfig default fields:')
for attr in sorted(dir(config)):
    if not attr.startswith('_'):
        try:
            val = getattr(config, attr)
            if not callable(val):
                print(f'  {attr:30s} = {val}')
        except Exception:
            pass

In [ ]:
# Cria engine de inferência (que também treina via train_text)
engine = nsos_ext.InferenceEngine()

# Carrega config padrão (40M mamba). Você pode trocar por hybrid_small/etc
# se quiser scale maior — mas T4 com tempo de sessão ≤12h aguenta 40-80M
MODEL_NAME = os.environ.get('NSOS_MODEL_NAME', 'mamba_small')
print(f'Carregando arquitetura: {MODEL_NAME}')

ok = engine.load_model(MODEL_NAME, config)
print(f'load_model OK: {ok}')

mem = engine.get_memory_usage()
print(f'Memory usage: {mem}')

## 7. Pre-training loop — train_text() em cada doc

Loop simples: para cada doc do corpus, chama `engine.train_text(text)`. O runtime C++ faz tokenização, forward, backward, optimizer step internamente.

Default smoke: **1500 steps** (~30-60 min em T4). Para produção real seria 100k+ steps.

In [ ]:
import time
import random
import matplotlib.pyplot as plt

MAX_STEPS = int(os.environ.get('NSOS_MAX_STEPS', '1500'))
LOG_EVERY = 25
EVAL_PROMPTS = [
    'O Imposto de Renda é',
    'O contribuinte deve declarar',
    'A nota fiscal eletrônica',
]

print(f'Pre-training: {MAX_STEPS} steps, log a cada {LOG_EVERY}')
print(f'Corpus path:  {CORPUS_PATH}')

# Carrega corpus em memória como lista de textos (corpus pode ser grande;
# em produção usaria streaming)
all_docs = []
with open(CORPUS_PATH, encoding='utf-8') as f:
    for line in f:
        try:
            all_docs.append(json.loads(line)['text'])
        except (json.JSONDecodeError, KeyError):
            continue
print(f'Total docs no corpus em memória: {len(all_docs):,}')

# Shuffle determinístico
random.seed(1337)
random.shuffle(all_docs)

losses = []
elapsed_per_step = []
t_start = time.time()
doc_idx = 0

for step in range(1, MAX_STEPS + 1):
    t0 = time.time()
    doc = all_docs[doc_idx % len(all_docs)]
    doc_idx += 1
    try:
        # train_text retorna loss (float) ou dict — depende da versão do binding
        result = engine.train_text(doc[:4096])  # cap em 4k chars pra evitar OOM
        loss = result if isinstance(result, (int, float)) else result.get('loss', float('nan'))
    except Exception as e:
        print(f'[step {step}] train_text falhou: {e}')
        continue
    elapsed_per_step.append(time.time() - t0)
    losses.append(float(loss))

    if step % LOG_EVERY == 0 or step == 1:
        recent_loss = sum(losses[-LOG_EVERY:]) / min(LOG_EVERY, len(losses))
        avg_step_s = sum(elapsed_per_step[-LOG_EVERY:]) / min(LOG_EVERY, len(elapsed_per_step))
        eta_min = (MAX_STEPS - step) * avg_step_s / 60
        print(f'[step {step:>5d}/{MAX_STEPS}] '
              f'loss={recent_loss:.3f}  step={avg_step_s:.2f}s  eta={eta_min:.1f}min')

total_min = (time.time() - t_start) / 60
print(f'\nDONE: {len(losses)} steps em {total_min:.1f} min')

## 8. Plot loss curve

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(losses, alpha=0.4, label='step loss')
# Smooth
window = max(1, len(losses) // 30)
if len(losses) > window:
    smoothed = [sum(losses[max(0, i-window):i+1]) / min(i+1, window) for i in range(len(losses))]
    plt.plot(smoothed, color='red', linewidth=2, label=f'smoothed (w={window})')
plt.xlabel('step')
plt.ylabel('loss')
plt.title(f'Oxta Contábil — continued pre-training ({MODEL_NAME}, {len(losses)} steps)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
out_png = DATA_ROOT / 'oxta_contabil_pretrain_loss.png'
plt.savefig(out_png, dpi=110)
plt.show()
print(f'Saved: {out_png}')

## 9. Geração de amostra (qualitativa)

Não é Q&A — só vê se o modelo aprendeu a continuar texto fiscal.

In [ ]:
gen_opts = nsos_ext.GenerationOptions()
gen_opts.max_tokens = 80
gen_opts.temperature = 0.7
gen_opts.top_p = 0.9

for prompt in EVAL_PROMPTS:
    try:
        out = engine.generate(prompt, gen_opts)
    except Exception as e:
        out = f'(falhou: {e})'
    print(f'\nPROMPT: {prompt!r}')
    print(f'CONT:   {out}')

## 10. Salvar pack em Drive (re-usável em sessões futuras)

In [ ]:
from datetime import datetime

PACK_DIR = Path('/content/drive/MyDrive/oxta_contabil_packs') / \
           f'pretrain_{MODEL_NAME}_{datetime.now().strftime("%Y%m%d_%H%M")}'
PACK_DIR.mkdir(parents=True, exist_ok=True)

try:
    engine.save_model_pack(str(PACK_DIR))
    print(f'Pack salvo em: {PACK_DIR}')
    for f in sorted(PACK_DIR.rglob('*')):
        if f.is_file():
            print(f'  {f.relative_to(PACK_DIR)}: {f.stat().st_size/1024**2:.2f} MB')
except Exception as e:
    print(f'save_model_pack falhou: {e}')
    print('Fallback: usa save_checkpoint')
    try:
        engine.save_checkpoint(str(PACK_DIR / 'checkpoint.nsos'))
        print(f'Checkpoint salvo')
    except Exception as e2:
        print(f'save_checkpoint também falhou: {e2}')

## 11. Próximos passos

### O que esse notebook prova:
- ✓ NSOS C++ builda em Colab e expõe `train_text` via Python
- ✓ Os 4 datasets brasileiros baixam, extraem texto cru, geram corpus único
- ✓ Loss desce durante pre-training (se desceu — verifique o plot acima)
- ✓ Pack salvo em Drive, reusável

### O que esse notebook NÃO prova (e precisa ser feito depois):
- **Que 1500 steps é suficiente.** Não é. Pre-training real precisa de 50k-500k+ steps.
  Para um modelo 40M com 3GB de corpus, 30k steps daria 1-2 epochs.
- **Que o modelo agora responde perguntas.** Pre-training só ensina linguagem,
  não responde perguntas — pra isso precisa SFT depois (próximo notebook).
- **Que a arquitetura escolhida (40M mamba_small) é a final.** Provavelmente
  vai precisar escalar pra 400M-1B antes do produto.

### Para escalar pra produção:
1. **Mais dados:** rodar `download_datasets.py --target-bytes 30GB` (CulturaX)
2. **Mais steps:** `NSOS_MAX_STEPS=50000` antes de Run All
3. **Modelo maior:** `NSOS_PARAM_SCALE=80m` na cell de bootstrap
4. **Sessão A100** se disponível — 3-5× mais rápido que T4
5. Quando loss estabilizar, salvar pack final e partir pro SFT (Q&A)

### Para o próximo notebook (SFT):
Carregar o pack salvo aqui via `engine.load_model_pack(...)`, e treinar em
Q&A pairs do BR-TaxQA-R formatados como ChatML — como o `oxta_contabil_smoke_train.ipynb`
fez com GPT-2 mas agora com nosso modelo já fluente em PT-BR fiscal.